In [1]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


In [2]:
documents = [
    Document(page_content='this is page 1'),
    Document(page_content='this is page 2'),
    Document(page_content='this is page 3'),
    Document(page_content='this is page 4'),    
]

In [3]:
embedding_model = OpenAIEmbeddings()

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name='my_collection'
)

# this creates a vector store with the following documnets and collection name

NameError: name 'embedding_model' is not defined

In [ ]:
retriver = vector_store.as_retriever(search_type = 'similarity',search_kwargs = {"k" : 2})
# we are creating a retriever from a vector store , k=2 , means 2 relevant embeddings will be returned

In [ ]:
query = "What is chroma used for?"
results = retriver.invoke(query)

In [5]:
# WHY did we use retriever when we can just use vector_store.similarity_search??????
# # because similarity_search uses only one type of metric for getting the docs
# # we can create even advanced retriver which uses different logic to retrieve the docs

In [ ]:
retriver = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = { "k" : 2 , "lambda_mult" : 1}
    # lambda tell us how diverse the ans will be , it ranges from 0-1 , 1 -> same as similarity search , 0 -> too diverse ans
)

In [ ]:
# what normal similiarity search does is if u give it a query , it will give u lets say k = 3 , 3 documents matching the query , so most likely all 3 documents will talk on the same topic right 
# what MMR does is , it will give u 3 topics but it wont be on the same topic , it will give u 3 different perspectives. 

In [ ]:
# # MultiQuery Retriever

# what this does it , it resolves the ambiguity of the query given by the user ,
# if the user given query is ambigious , we can use multi query retriever for this

# multi query retriever gives the query to a LLM , the LLM generates lets say 5 more similar queries , and we get the similar documents from 
# the 5 queries each from the vector store , therefore we have 5 documents now , out of them top "k" documents are returned.


In [7]:
from langchain.retrievers.multi_query import MultiQueryRetriever

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vector_store.as_retriever(search_kwargs = {"k" : 5}),
    # this retriever is used to get similar documents for those new 5 queries from the LLM
    llm= ChatOpenAI(model = 'gpt-3.5-turbo')
)

query = "how to improve energy levels"

In [ ]:
results = multiquery_retriever.invoke(query)